# Queueing Theory Validation Checks

This notebook validates the simulator against classical queueing models with known analytic formulas.

The goal is to confirm that the event engine, service logic, and abandonment logic reproduce standard theory before using the simulator for policy experiments.

## What is being tested

We compare simulation output with theory for four benchmark models:

- M/M/1
- M/M/N
- M/M/1+M
- M/M/N+M

These tests check both stable queues without abandonment and Erlang-A systems with abandonment.

The next cells define helper functions for:

- empirical metrics extracted from simulation output
- exact formulas for M/M/1
- exact formulas for M/M/N via Erlang-C
- stationary truncation formulas for Erlang-A systems

In [21]:
import math
import numpy as np
import pandas as pd

from gi_gi_n_gi_multiclass.core.engine import run_sim
from gi_gi_n_gi_multiclass.core.types import SimConfig
from gi_gi_n_gi_multiclass.dists.common import Exponential
from gi_gi_n_gi_multiclass.policies.fcfs import FCFS
from gi_gi_n_gi_multiclass.policies.lcfs import LCFS
from gi_gi_n_gi_multiclass.policies.tiq import TIQ
from gi_gi_n_gi_multiclass.outputs.schemas import customers_to_dataframe
from gi_gi_n_gi_multiclass.metrics.steady_state import time_average_L
from gi_gi_n_gi_multiclass.metrics.basic import summary_table


In [22]:
# --- Helper: empirical metrics from a SimResult (post-warmup arrivals) ---

def empirical_metrics(res):
    df = customers_to_dataframe(res)
    t0 = res.config.warmup_time
    df = df[df["arrival_time"] >= t0].copy()

    served = df[df["outcome"] == "SERVED"].copy()
    if len(served) == 0:
        raise RuntimeError("No served customers post-warmup; choose a stable regime or longer run.")

    served["Wq"] = served["service_start"] - served["arrival_time"]
    served["W"] = served["service_end"] - served["arrival_time"]

    # Probability an arrival waits: estimate among served customers (in stable no-abandon this is fine)
    P_wait = float((served["Wq"] > 1e-12).mean())

    # Time averages from the process (coarse grid approximation)
    stats = time_average_L(res)

    return {
        "lambda_hat": len(df) / max(1e-9, (res.end_time - t0)),
        "Wq_emp": float(served["Wq"].mean()),
        "W_emp": float(served["W"].mean()),
        "P_wait_emp": P_wait,
        "L_emp": float(stats["L"]),
        "Lq_emp": float(stats["Lq"]),
        "served_count": int(len(served)),
    }

## Test 1: M/M/1

We first validate the simulator in the simplest single-server setting without abandonment.

This checks the most basic queueing quantities against exact formulas:

$$
L, \quad L_q, \quad W, \quad W_q, \quad P(\text{wait}>0).
$$

In [23]:
# --- Theory: M/M/1 exact formulas ---

def theory_mm1(lam, mu):
    if not (lam < mu):
        raise ValueError("M/M/1 requires lam < mu for stability.")
    rho = lam / mu
    L = rho / (1.0 - rho)
    Lq = rho * rho / (1.0 - rho)
    W = 1.0 / (mu - lam)
    Wq = lam / (mu * (mu - lam))
    P_wait = rho  # P(Wq>0)
    return {
        "rho": rho,
        "L_th": L,
        "Lq_th": Lq,
        "W_th": W,
        "Wq_th": Wq,
        "P_wait_th": P_wait,
    }

In [24]:
# --- Theory: M/M/N exact formulas (Erlang-C) ---

def theory_mmn(lam, mu, N):
    if N <= 0:
        raise ValueError("N must be positive.")
    a = lam / mu
    rho = lam / (N * mu)
    if not (rho < 1.0):
        raise ValueError("M/M/N requires rho < 1 for stability (lam < N*mu).")

    # Compute P0
    s = 0.0
    for k in range(N):
        s += (a ** k) / math.factorial(k)
    s += (a ** N) / math.factorial(N) * (1.0 / (1.0 - rho))
    P0 = 1.0 / s

    # Erlang-C (probability of delay)
    P_wait = (a ** N) / math.factorial(N) * (1.0 / (1.0 - rho)) * P0

    # Mean queue length, waits
    Lq = P_wait * rho / (1.0 - rho)
    Wq = Lq / lam
    W = Wq + 1.0 / mu
    L = lam * W

    return {
        "rho": rho,
        "P0": P0,
        "P_wait_th": P_wait,
        "L_th": L,
        "Lq_th": Lq,
        "W_th": W,
        "Wq_th": Wq,
    }

In [25]:
# --- A) M/M/1 validation run ---

lam = 0.8
mu = 1.0
N = 1

cfg = SimConfig(
    n_servers=N,
    interarrival=Exponential(rate=lam),
    service=Exponential(rate=mu),
    patience=None,         # no abandonment
    policy=FCFS(),
    seed=123,
    warmup_time=500.0,
    run_time=5000.0,
    collect_event_log=False,
)

res = run_sim(cfg)

th = theory_mm1(lam, mu)
emp = empirical_metrics(res)

mm1_compare = pd.DataFrame([{
    "model": "M/M/1",
    "lam": lam,
    "mu": mu,
    "rho": th["rho"],
    "Wq_th": th["Wq_th"],
    "Wq_emp": emp["Wq_emp"],
    "W_th": th["W_th"],
    "W_emp": emp["W_emp"],
    "Lq_th": th["Lq_th"],
    "Lq_emp": emp["Lq_emp"],
    "L_th": th["L_th"],
    "L_emp": emp["L_emp"],
    "P_wait_th": th["P_wait_th"],
    "P_wait_emp": emp["P_wait_emp"],
    "lambda_hat": emp["lambda_hat"],
    "served_count": emp["served_count"],
}])

mm1_compare

,model,lam,mu,rho,Wq_th,Wq_emp,W_th,W_emp,Lq_th,Lq_emp,L_th,L_emp,P_wait_th,P_wait_emp,lambda_hat,served_count
0,M/M/1,0.8,1.0,0.8,4.0,4.028057,5.0,5.015051,3.2,3.223652,4.0,4.012757,0.8,0.782674,0.8006,3994


## Test 2: M/M/N

We next move to the multi-server case without abandonment.

This validates the simulator against Erlang-C theory, which checks that multi-server queueing and service assignment are implemented correctly.

In [26]:
# --- B) M/M/N validation run (Erlang-C) ---

lam = 8.0
mu = 1.0
N = 10

cfg = SimConfig(
    n_servers=N,
    interarrival=Exponential(rate=lam),
    service=Exponential(rate=mu),
    patience=None,         # no abandonment
    policy=FCFS(),
    seed=123,
    warmup_time=500.0,
    run_time=5000.0,
    collect_event_log=False,
)

res = run_sim(cfg)

th = theory_mmn(lam, mu, N)
emp = empirical_metrics(res)

mmn_compare = pd.DataFrame([{
    "model": f"M/M/{N}",
    "lam": lam,
    "mu": mu,
    "rho": th["rho"],
    "Wq_th": th["Wq_th"],
    "Wq_emp": emp["Wq_emp"],
    "W_th": th["W_th"],
    "W_emp": emp["W_emp"],
    "Lq_th": th["Lq_th"],
    "Lq_emp": emp["Lq_emp"],
    "L_th": th["L_th"],
    "L_emp": emp["L_emp"],
    "P_wait_th": th["P_wait_th"],
    "P_wait_emp": emp["P_wait_emp"],
    "lambda_hat": emp["lambda_hat"],
    "served_count": emp["served_count"],
}])

mmn_compare

,model,lam,mu,rho,Wq_th,Wq_emp,W_th,W_emp,Lq_th,Lq_emp,L_th,L_emp,P_wait_th,P_wait_emp,lambda_hat,served_count
0,M/M/10,8.0,1.0,0.8,0.20459,0.183521,1.20459,1.180727,1.636721,1.469382,9.636721,9.452184,0.40918,0.395499,8.0066,40033


In [27]:
# --- B) M/M/N validation run (Erlang-C) ---

lam = 8.0
mu = 1.0
N = 10

cfg = SimConfig(
    n_servers=N,
    interarrival=Exponential(rate=lam),
    service=Exponential(rate=mu),
    patience=None,         # no abandonment
    policy=FCFS(),
    seed=123,
    warmup_time=500.0,
    run_time=5000.0,
    collect_event_log=False,
)

res = run_sim(cfg)

th = theory_mmn(lam, mu, N)
emp = empirical_metrics(res)

mmn_compare = pd.DataFrame([{
    "model": f"M/M/{N}",
    "lam": lam,
    "mu": mu,
    "rho": th["rho"],
    "Wq_th": th["Wq_th"],
    "Wq_emp": emp["Wq_emp"],
    "W_th": th["W_th"],
    "W_emp": emp["W_emp"],
    "Lq_th": th["Lq_th"],
    "Lq_emp": emp["Lq_emp"],
    "L_th": th["L_th"],
    "L_emp": emp["L_emp"],
    "P_wait_th": th["P_wait_th"],
    "P_wait_emp": emp["P_wait_emp"],
    "lambda_hat": emp["lambda_hat"],
    "served_count": emp["served_count"],
}])

mmn_compare

,model,lam,mu,rho,Wq_th,Wq_emp,W_th,W_emp,Lq_th,Lq_emp,L_th,L_emp,P_wait_th,P_wait_emp,lambda_hat,served_count
0,M/M/10,8.0,1.0,0.8,0.20459,0.183521,1.20459,1.180727,1.636721,1.469382,9.636721,9.452184,0.40918,0.395499,8.0066,40033


## Test 3: M/M/1+M

We now add abandonment to the single-server system.

This is the simplest Erlang-A benchmark and checks whether abandonment events are scheduled and removed correctly.

In [28]:
# --- Empirical rates and L from simulation (post-warmup) ---

def empirical_rates_and_L(res):
    df = customers_to_dataframe(res)
    t0 = res.config.warmup_time
    T = max(1e-9, (res.end_time - t0))

    df = df[df["arrival_time"] >= t0].copy()

    served = df[df["outcome"] == "SERVED"]
    abandoned = df[df["outcome"] == "ABANDONED"]

    lam_hat = len(df) / T
    served_rate = len(served) / T
    abandon_rate = len(abandoned) / T

    # time-average L and Lq from grid sampling
    stats = time_average_L(res)

    return {
        "lambda_hat": lam_hat,
        "served_rate_emp": served_rate,
        "abandon_rate_emp": abandon_rate,
        "L_emp": float(stats["L"]),
        "Lq_emp": float(stats["Lq"]),
        "arrivals": int(len(df)),
        "served": int(len(served)),
        "abandoned": int(len(abandoned)),
        "T_window": T,
    }

In [29]:
def erlangA_stationary(lam, mu, theta, N, n_max=20000, tail_tol=1e-14):
    if lam <= 0 or mu <= 0 or theta <= 0:
        raise ValueError("lam, mu, theta must be > 0")
    if N <= 0:
        raise ValueError("N must be > 0")

    pi = np.zeros(n_max + 1, dtype=float)
    pi[0] = 1.0
    for n in range(1, n_max + 1):
        delta = mu * min(n, N) + theta * max(n - N, 0)
        pi[n] = pi[n - 1] * (lam / delta)
        # optional early stop if terms are tiny for a while
        if n > N + 50 and pi[n] < tail_tol:
            # still keep a few more to avoid premature stop
            pass

    Z = float(np.sum(pi))
    if not np.isfinite(Z) or Z <= 0:
        raise RuntimeError("Normalization failed; increase n_max or check parameters.")

    pi /= Z

    # Means
    n_vals = np.arange(n_max + 1, dtype=float)
    L = float(np.sum(n_vals * pi))

    # Busy servers E[min(n,N)]
    busy = float(np.sum(np.minimum(n_vals, float(N)) * pi))

    # Queue E[max(n-N,0)]
    Lq = float(np.sum(np.maximum(n_vals - float(N), 0.0) * pi))

    # Rates
    served_rate = mu * busy
    abandon_rate = theta * Lq

    return {
        "L_th": L,
        "Lq_th": Lq,
        "served_rate_th": served_rate,
        "abandon_rate_th": abandon_rate,
        "P0": float(pi[0]),
        "pi": pi,  # returned for debugging if needed
    }

In [31]:
# --- 3) M/M/1+M validation run ---

lam = 2.0
mu = 1.0
theta = 0.5
N = 1

cfg = SimConfig(
    n_servers=N,
    interarrival=Exponential(rate=lam),
    service=Exponential(rate=mu),
    patience=Exponential(rate=theta),
    policy=FCFS(),          # ordering doesn't matter for M/M/1+M due to memoryless
    seed=123,
    warmup_time=500.0,
    run_time=5000.0,
    collect_event_log=False,
)

res = run_sim(cfg)

th = erlangA_stationary(lam, mu, theta, N, n_max=1000)
emp = empirical_rates_and_L(res)

mm1m_compare = pd.DataFrame([{
    "model": "M/M/1+M",
    "lam": lam,
    "mu": mu,
    "theta": theta,
    "L_th": th["L_th"],
    "L_emp": emp["L_emp"],
    "Lq_th": th["Lq_th"],
    "Lq_emp": emp["Lq_emp"],
    "served_rate_th": th["served_rate_th"],
    "served_rate_emp": emp["served_rate_emp"],
    "abandon_rate_th": th["abandon_rate_th"],
    "abandon_rate_emp": emp["abandon_rate_emp"],
    "flow_th": th["served_rate_th"] + th["abandon_rate_th"],
    "flow_emp": emp["served_rate_emp"] + emp["abandon_rate_emp"],
    "lambda_hat": emp["lambda_hat"],
    "arrivals": emp["arrivals"],
    "served": emp["served"],
    "abandoned": emp["abandoned"],
}])

mm1m_compare

,model,lam,mu,theta,L_th,L_emp,Lq_th,Lq_emp,served_rate_th,served_rate_emp,abandon_rate_th,abandon_rate_emp,flow_th,flow_emp,lambda_hat,arrivals,served,abandoned
0,M/M/1+M,2.0,1.0,0.5,3.074629,3.012278,2.149259,2.086362,0.925371,0.9496,1.074629,1.0566,2.0,2.0062,2.0064,10032,4748,5283


## Test 4: M/M/N+M

Finally we validate the many-server Erlang-A model.

This is the most important abandonment benchmark in the notebook because it combines:

- multiple servers
- queueing
- abandonment
- overload stabilization

In [33]:
# --- 4) M/M/N+M validation run (choose stable or overloaded; Erlang-A is stable either way) ---

lam = 20.0
mu = 1.0
theta = 0.5
N = 10

cfg = SimConfig(
    n_servers=N,
    interarrival=Exponential(rate=lam),
    service=Exponential(rate=mu),
    patience=Exponential(rate=theta),
    policy=FCFS(),          # for memoryless patience + service, ordering has limited impact on stationary totals
    seed=123,
    warmup_time=500.0,
    run_time=5000.0,
    collect_event_log=False,
)

res = run_sim(cfg)

th = erlangA_stationary(lam, mu, theta, N, n_max=1000)
emp = empirical_rates_and_L(res)

mmnm_compare = pd.DataFrame([{
    "model": f"M/M/{N}+M",
    "lam": lam,
    "mu": mu,
    "theta": theta,
    "L_th": th["L_th"],
    "L_emp": emp["L_emp"],
    "Lq_th": th["Lq_th"],
    "Lq_emp": emp["Lq_emp"],
    "served_rate_th": th["served_rate_th"],
    "served_rate_emp": emp["served_rate_emp"],
    "abandon_rate_th": th["abandon_rate_th"],
    "abandon_rate_emp": emp["abandon_rate_emp"],
    "flow_th": th["served_rate_th"] + th["abandon_rate_th"],
    "flow_emp": emp["served_rate_emp"] + emp["abandon_rate_emp"],
    "lambda_hat": emp["lambda_hat"],
    "arrivals": emp["arrivals"],
    "served": emp["served"],
    "abandoned": emp["abandoned"],
}])

mmnm_compare

,model,lam,mu,theta,L_th,L_emp,Lq_th,Lq_emp,served_rate_th,served_rate_emp,abandon_rate_th,abandon_rate_emp,flow_th,flow_emp,lambda_hat,arrivals,served,abandoned
0,M/M/10+M,20.0,1.0,0.5,30.000271,30.136433,20.000542,20.136546,9.999729,9.9594,10.000271,10.0934,20.0,20.0528,20.0576,100288,49797,50467


In [34]:
# --- Optional: combine 3) and 4) and compute relative errors ---

def relerr(emp, th):
    if th == 0.0:
        return float("nan")
    return (emp - th) / th

both = pd.concat([mm1m_compare, mmnm_compare], ignore_index=True)

for metric in ["L", "Lq", "served_rate", "abandon_rate", "flow"]:
    both[f"{metric}_relerr"] = [
        relerr(both.loc[i, f"{metric}_emp"], both.loc[i, f"{metric}_th"])
        for i in range(len(both))
    ]

both[[
    "model",
    "L_th", "L_emp", "L_relerr",
    "Lq_th", "Lq_emp", "Lq_relerr",
    "served_rate_th", "served_rate_emp", "served_rate_relerr",
    "abandon_rate_th", "abandon_rate_emp", "abandon_rate_relerr",
    "flow_th", "flow_emp", "flow_relerr",
    "lambda_hat", "arrivals", "served", "abandoned",
]]

,model,L_th,L_emp,L_relerr,Lq_th,Lq_emp,Lq_relerr,served_rate_th,served_rate_emp,served_rate_relerr,abandon_rate_th,abandon_rate_emp,abandon_rate_relerr,flow_th,flow_emp,flow_relerr,lambda_hat,arrivals,served,abandoned
0,M/M/1+M,3.074629,3.012278,-0.020279,2.149259,2.086362,-0.029264,0.925371,0.9496,0.026184,1.074629,1.0566,-0.016777,2.0,2.0062,0.00310,2.0064,10032,4748,5283
1,M/M/10+M,30.000271,30.136433,0.004539,20.000542,20.136546,0.006800,9.999729,9.9594,-0.004033,10.000271,10.0934,0.009313,20.0,20.0528,0.00264,20.0576,100288,49797,50467


## Policy invariance check for Erlang-A

In an M/M/N+M system with exponential service and exponential patience, aggregate steady-state quantities should not depend strongly on the non-idling queue discipline.

To check this, we compare:

- FCFS
- LCFS
- TIQ with arbitrary thresholds

The expectation is that the overall rates and time-average queue lengths should be nearly identical across policies.

In [35]:
# --- Erlang-A parameters ---
lam = 20.0
mu = 1.0
theta = 0.5
N = 10

warmup_time = 500.0
run_time = 5000.0

interarrival = Exponential(rate=lam)
service = Exponential(rate=mu)
patience = Exponential(rate=theta)

policies = {
    "FCFS": FCFS(),
    "LCFS": LCFS(),
    "TIQ_0.5_1.0": TIQ(w_l=0.5, w_h=1.0),
    "TIQ_0.2_2.0": TIQ(w_l=0.2, w_h=2.0),
}

# Multiple seeds to show invariance robustly
seeds = [1, 2, 3, 4, 5]

rows = []
for pol_name, pol in policies.items():
    for seed in seeds:
        cfg = SimConfig(
            n_servers=N,
            interarrival=interarrival,
            service=service,
            patience=patience,
            policy=pol,
            seed=seed,
            warmup_time=warmup_time,
            run_time=run_time,
            collect_event_log=False,
        )
        res = run_sim(cfg)

        tbl = summary_table(res)
        stats = time_average_L(res)

        tbl["L_timeavg"] = stats["L"]
        tbl["Lq_timeavg"] = stats["Lq"]
        tbl["policy_name"] = pol_name
        tbl["seed"] = seed
        rows.append(tbl)

out = pd.concat(rows, ignore_index=True)

cols = ["policy_name", "seed", "lambda_hat", "served_rate", "abandon_rate", "Lq_timeavg", "L_timeavg"]
out[cols].sort_values(["policy_name", "seed"])

,policy_name,seed,lambda_hat,served_rate,abandon_rate,Lq_timeavg,L_timeavg
0,FCFS,1,20.1266,9.9046,10.2194,20.414851,30.414779
1,FCFS,2,20.0362,10.0178,10.0146,19.863754,29.863586
2,FCFS,3,19.9816,9.9806,9.9958,19.977877,29.977406
3,FCFS,4,19.9738,9.9668,10.0034,19.883707,29.883306
4,FCFS,5,20.0052,9.9960,10.0058,19.966210,29.965923
5,LCFS,1,20.1266,9.9618,10.1618,20.270566,30.270358
6,LCFS,2,20.0362,10.0018,10.0302,20.101232,30.101067
7,LCFS,3,19.9816,9.9348,10.0416,20.005718,30.005569
8,LCFS,4,19.9738,9.9604,10.0096,19.960297,29.960150
9,LCFS,5,20.0052,10.0554,9.9466,19.899866,29.899760


In [36]:
# Summarize by policy: mean and std across seeds
summary = (
    out.groupby("policy_name")[["lambda_hat", "served_rate", "abandon_rate", "Lq_timeavg", "L_timeavg"]]
      .agg(["mean", "std"])
)
summary

lambda_hat           served_rate           abandon_rate            \
                  mean       std        mean       std         mean       std   
policy_name                                                                     
FCFS          20.02468  0.061937     9.97316  0.042751     10.04780  0.096162   
LCFS          20.02468  0.061937     9.98284  0.047113     10.03796  0.078359   
TIQ_0.2_2.0   20.02468  0.061937     9.97284  0.029888     10.04840  0.076963   
TIQ_0.5_1.0   20.02468  0.061937     9.96736  0.049081     10.05348  0.104153   

            Lq_timeavg            L_timeavg            
                  mean       std       mean       std  
policy_name                                            
FCFS         20.021280  0.225586  30.021000  0.225685  
LCFS         20.047536  0.144734  30.047381  0.144698  
TIQ_0.2_2.0  19.976030  0.163644  29.975697  0.163699  
TIQ_0.5_1.0  20.026569  0.238919  30.026327  0.238952

## Takeaway

This notebook validates the simulator against standard queueing theory benchmarks.

If the empirical and theoretical quantities agree closely, then the package is correctly handling:

- single-server and multi-server queueing
- waiting-time dynamics
- abandonment
- steady-state flow balance

These checks provide the theoretical foundation for the later policy notebooks.